# 01 · EDA: FWC bear calls (Phase 0)

Run `python -m bearcast.ingest.fwc --all` and `python -m bearcast.ingest.gbif` first.

**Goal of this notebook:** understand the target data well enough to decide
(1) the start year, (2) the time step, (3) the spatial unit, and (4) which call types count as "conflict".
Write your answers in the last cell.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 60)
RAW = Path("../data/raw") if Path("../data/raw").exists() else Path("data/raw")
FL_BBOX = dict(minx=-87.7, maxx=-79.8, miny=24.4, maxy=31.1)  # rough Florida extent
TZ = "America/New_York"  # most of FL; the western Panhandle is Central time

calls = gpd.read_parquet(RAW / "fwc_bear_calls.parquet")
print(json.dumps(json.loads((RAW / "fwc_bear_calls.manifest.json").read_text()), indent=2)[:800])
calls.head()

## 1. Schema and missing values

In [ ]:
print(calls.shape)
display(calls.dtypes.to_frame("dtype").assign(
    n_null=calls.isna().sum(),
    pct_null=(calls.isna().mean() * 100).round(2),
    n_unique=calls.nunique(dropna=True),
))

## 2. Duplicates
`OBJECTID` should be unique. `IncidentID` and repeated date + location are the ones to watch.
One incident can produce several calls, and the same bear can be reported many times.

In [ ]:
print("OBJECTID unique:", calls["OBJECTID"].is_unique)
if "IncidentID" in calls:
    dup_inc = calls["IncidentID"].dropna().duplicated(keep=False)
    print("Rows sharing an IncidentID:", int(dup_inc.sum()))
key = ["ReportDate", "Latitude", "Longitude", "CallNature"]
key = [k for k in key if k in calls]
print("Rows duplicated on", key, ":", int(calls.duplicated(subset=key, keep=False).sum()))

## 3. Time coverage
ArcGIS stores dates in UTC. Convert to local time **before** taking the date, or calls near midnight shift a day.

In [ ]:
calls["report_local"] = calls["ReportDate"].dt.tz_convert(TZ)
calls["date"] = calls["report_local"].dt.date
calls["year"] = calls["report_local"].dt.year
calls["month"] = calls["report_local"].dt.month

print("Range:", calls["report_local"].min(), "->", calls["report_local"].max())
print("Missing dates:", int(calls["ReportDate"].isna().sum()))
# Are times real or just midnight placeholders?
t = calls["report_local"].dt.time.astype(str)
print("Share of records at exactly 00:00 local:", round((t == "00:00:00").mean(), 3))

per_year = calls.groupby("year").size()
ax = per_year.plot(kind="bar", figsize=(14, 4), color="#4c6a92", title="Calls per year")
ax.set_xlabel(""); plt.tight_layout(); plt.show()

**Look for:** sudden jumps that track changes in *reporting* (new hotline, online form, database migration) rather than bears.
Those years may need to be dropped or modeled with a regime indicator.

## 4. Seasonality

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
calls.groupby("month").size().plot(kind="bar", ax=axes[0], color="#4c6a92", title="Calls by month (all years)")
heat = calls.pivot_table(index="year", columns="month", values="OBJECTID", aggfunc="count").fillna(0)
im = axes[1].imshow(heat.values, aspect="auto", cmap="viridis")
axes[1].set_yticks(range(len(heat.index))[::3], heat.index[::3]); axes[1].set_xticks(range(12), range(1, 13))
axes[1].set_title("Calls: year x month"); fig.colorbar(im, ax=axes[1])
plt.tight_layout(); plt.show()

## 5. Call types (`CallNature`, free text)
There's no coded domain, so expect inconsistent spellings. Build a mapping to a small set of classes
(e.g. *garbage/attractant*, *in yard/sighting*, *property damage*, *pets/livestock*, *road/vehicle*, *other*).

In [ ]:
cn = calls["CallNature"].fillna("<missing>")
print("Distinct raw values:", cn.nunique())
norm = cn.str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
print("Distinct after normalizing case/whitespace:", norm.nunique())
norm.value_counts().head(40)

In [ ]:
# Does the mix of call types change over time? (Reporting practice vs. behavior)
top = norm.value_counts().head(8).index
mix = (calls.assign(cn=norm.where(norm.isin(top), "other"))
            .pivot_table(index="year", columns="cn", values="OBJECTID", aggfunc="count").fillna(0))
mix.div(mix.sum(axis=1), axis=0).plot.area(figsize=(14, 5), title="Share of call types by year", cmap="tab10")
plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left"); plt.tight_layout(); plt.show()

## 6. Spatial quality

In [ ]:
geom_xy = np.c_[calls.geometry.x, calls.geometry.y]
if {"Latitude", "Longitude"} <= set(calls.columns):
    d = np.hypot(geom_xy[:, 0] - calls["Longitude"], geom_xy[:, 1] - calls["Latitude"])
    print("Geometry vs Lat/Lon attribute disagreement > 0.001 deg:", int((d > 1e-3).sum()))
print("Empty / null geometries:", int(calls.geometry.is_empty.sum() + calls.geometry.isna().sum()))
outside = ~calls.geometry.x.between(FL_BBOX["minx"], FL_BBOX["maxx"]) | ~calls.geometry.y.between(FL_BBOX["miny"], FL_BBOX["maxy"])
print("Points outside Florida bbox:", int(outside.sum()))

# Coordinate precision: few decimals => geocoded to a city/ZIP centroid, not the real spot.
def decimals(x):
    s = f"{x:.10f}".rstrip("0")
    return len(s.split(".")[1]) if "." in s else 0
prec = calls.geometry.x.dropna().map(decimals)
print("\nDecimal places in longitude:"); print(prec.value_counts().sort_index())
stacked = calls.geometry.to_wkb().value_counts()
print("\nLocations with >= 20 calls at the exact same point:", int((stacked >= 20).sum()))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
rng_path = RAW / "fwc_range.parquet"
if rng_path.exists():
    rng_gdf = gpd.read_parquet(rng_path)
    shades = {"frequent": "#1b5e20", "common": "#43a047", "occasional": "#a5d6a7", "rare": "#e8f5e9"}
    for label, sub in rng_gdf.groupby("Label"):
        sub.plot(ax=ax, color=shades.get(str(label).strip().lower(), "#cccccc"), alpha=0.35, edgecolor="#2e7d32", linewidth=0.3)
dec = (calls["year"] // 10 * 10).astype("Int64").astype(str)
for d_, sub in calls.loc[~outside].groupby(dec):
    ax.scatter(sub.geometry.x, sub.geometry.y, s=1, alpha=0.4, label=f"{d_}s")
ax.set_title("Bear calls by decade (range polygons in green)")
ax.legend(markerscale=10, loc="lower left"); ax.set_aspect("equal"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
calls["BMU"].fillna("<missing>").value_counts().plot(kind="barh", ax=axes[0], title="Calls by BMU")
calls["County"].fillna("<missing>").value_counts().head(20).plot(kind="barh", ax=axes[1], title="Top 20 counties")
plt.tight_layout(); plt.show()

## 7. Other FWC layers (quick look)

In [ ]:
for name in ["roadkill", "captures", "relocations", "telemetry_camp_blanding", "telemetry_combined"]:
    p = RAW / f"fwc_{name}.parquet"
    if not p.exists():
        print(f"{name}: not downloaded"); continue
    g = gpd.read_parquet(p)
    dcols = [c for c in g.columns if str(g[c].dtype).startswith("datetime64")]
    span = f"{g[dcols[0]].min()} -> {g[dcols[0]].max()}" if dcols else "no date column"
    print(f"{name:26s} rows={len(g):>8,}  cols={len(g.columns):>3}  {span}")

In [ ]:
p = RAW / "fwc_telemetry_camp_blanding.parquet"
if p.exists():
    tel = gpd.read_parquet(p)
    print("Bears:", tel["Bear"].nunique(), " Collars:", tel["Collar"].nunique())
    per_bear = tel.groupby("Bear").agg(n_fixes=("OBJECTID", "size"),
                                       first=("GMT_DATE", "min"), last=("GMT_DATE", "max"))
    display(per_bear.sort_values("n_fixes", ascending=False).head(15))
    print("VALIDATED values:", tel["VALIDATED"].value_counts(dropna=False).to_dict())

## 8. GBIF (non-iNaturalist) occurrences

In [ ]:
p = RAW / "gbif_ursus_americanus_fl.parquet"
if p.exists():
    occ = gpd.read_parquet(p)
    m = json.loads(p.with_suffix(".manifest.json").read_text())
    print(f"Kept {m['n_kept']} records; dropped {m['n_inaturalist_dropped']} iNaturalist records")
    display(occ["datasetName"].value_counts().head(15))
    display(occ["basisOfRecord"].value_counts())
    occ["year"].dropna().astype(int).value_counts().sort_index().plot(kind="bar", figsize=(14, 3), title="GBIF records per year")
    plt.tight_layout(); plt.show()
    print("License mix:", m["license_counts"])

## 9. Phase 0 decisions (fill these in)

| Decision | Your answer | Evidence from above |
|---|---|---|
| Start year for modeling | | Section 3: where does reporting look stable? |
| Time step (week / month) | | Section 4: is weekly too sparse per cell? |
| Spatial unit (grid size or H3 resolution) | | Section 6: coordinate precision limits how fine you can go |
| Which call types count as "conflict" | | Section 5 mapping |
| How to handle duplicates / repeat reports | | Section 2 |
| Exclude relocation points from presence data? | | Section 7 |
